In [ ]:
import os

val_root = "/kaggle/input/datasets/username/numerai-data-v52/validation.parquet"
weights_cb = "/kaggle/input/datasets/username/numerai-weights-v52-dual-c0102/catboost_model_0058.cbm"
weights_mlp = "/kaggle/input/datasets/username/numerai-weights-v52-dual-c0102/mlp_model_0102.pth"
feat_path = "/kaggle/input/datasets/username/numerai-custom-features/custom_features.json"

print(f"{val_root}, {weights_cb}, {weights_mlp}, {feat_path}")

/kaggle/input/datasets/shuangsong/numerai-data-v52/validation.parquet, /kaggle/input/datasets/shuangsong/numerai-weights-v52-dual-c0102/catboost_model_0058.cbm, /kaggle/input/datasets/shuangsong/numerai-weights-v52-dual-c0102/mlp_model_0102.pth, /kaggle/input/datasets/shuangsong/numerai-custom-features/custom_features.json


In [3]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import json
from catboost import CatBoostRegressor

cb_model = CatBoostRegressor()
cb_model.load_model(weights_cb)

class NumeraiMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(0.1), 
            nn.Linear(in_dim, 256),
            nn.SiLU(),     
            nn.Linear(256, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze()

In [6]:
with open(feat_path, 'r') as f:
    features = json.load(f)['feature_sets']['custom_features']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp = NumeraiMLP(len(features) + 1).to(device)
mlp.load_state_dict(torch.load(weights_mlp, map_location=device))
mlp.eval()

val_df = pd.read_parquet(val_root, columns=features)
X_val_np = val_df[features].fillna(0.5).values.astype(np.float32)

batch_size = 200000 
all_preds = []

with torch.no_grad():
    for i in range(0, len(X_val_np), batch_size):
        x_batch_np = X_val_np[i : i + batch_size]
        
        cb_preds = cb_model.predict(x_batch_np)
        cb_preds_t = torch.tensor(cb_preds, dtype=torch.float32).unsqueeze(1).to(device)
        
        x_batch_t = torch.from_numpy(x_batch_np).to(device)
        x_combined = torch.cat([x_batch_t, cb_preds_t], dim=1)
        
        batch_preds = mlp(x_combined).cpu().numpy().flatten()
        all_preds.append(batch_preds)
        
        if (i // batch_size) % 5 == 0:
            print(f"Finished: {i}/{len(X_val_np)}")

preds = np.concatenate(all_preds)
submission = pd.DataFrame({
    "id": val_df.index,  
    "prediction": pd.Series(preds).rank(pct=True).values
})
submission.to_csv("val_result_dual.csv", index=False)

Finished: 0/3943998
Finished: 1000000/3943998
Finished: 2000000/3943998
Finished: 3000000/3943998
